# May 25 — TFIM-QRC Light-Touch Prototype

This notebook runs the smallest end-to-end exact-state TFIM quantum reservoir prototype for the Phase 2 volatility-forecasting project.

Scope:

- target: `future_rv_20d`;
- fallback architecture: 6 qubits, PCA-6, 6 temporal anchors, Z-only exact expectations;
- readout: Ridge regression on log-volatility;
- metrics: RMSE, QLIKE, Mincer-Zarnowitz.

This is viability evidence and architecture validation, not a final performance benchmark.

In [ ]:
from pathlib import Path
import os

if Path.cwd().name == "notebooks":
    os.chdir("..")

import pandas as pd

from qpitome_qrc.data.features import FEATURE_COLUMNS
from qpitome_qrc.data.loaders import load_phase2_volatility_data
from qpitome_qrc.data.pca import fit_transform_pca_splits_train_only
from qpitome_qrc.data.splits import chronological_tabular_split
from qpitome_qrc.qrc.tfim_reservoir import (
    TFIMQRCConfig,
    fit_tfim_qrc_regressor,
    make_qrc_sequence_splits,
    summarize_qrc_result,
)

## 1. Load data and build PCA-6 sequence windows

In [ ]:
target = "future_rv_20d"

df = load_phase2_volatility_data()
splits = chronological_tabular_split(df)

pca6 = fit_transform_pca_splits_train_only(
    splits,
    feature_columns=FEATURE_COLUMNS,
    target_columns=[target],
    n_components=6,
    prefix="pca6",
)

sequence_splits = make_qrc_sequence_splits(
    pca6.splits,
    feature_columns=pca6.feature_columns,
    target_column=target,
    lookback_days=40,
)

pca6.explained_variance

In [ ]:
{name: (X.shape, y.shape) for name, (X, y, dates) in sequence_splits.items()}

## 2. Run fallback 6-qubit QRC prototype

In [ ]:
fallback_config = TFIMQRCConfig(
    qubits=6,
    pca_components=6,
    lookback_days=40,
    anchor_count=6,
    anchor_policy="even",
    observable_mode="z",
    trotter_steps_per_anchor=1,
    coupling_scale=0.7,
    transverse_field=0.5,
    evolution_time=0.5,
    ridge_alpha=10.0,
    target_transform="log",
    seed=42,
)

fallback_result = fit_tfim_qrc_regressor(
    sequence_splits,
    config=fallback_config,
    target=target,
    verbose=True,
)

fallback_summary = pd.DataFrame([summarize_qrc_result(fallback_result)])
fallback_summary.T

## 3. Optional small observable probe

In [ ]:
probe_rows = []

for observable_mode in ["z", "zx", "zxzz"]:
    config = TFIMQRCConfig(
        qubits=6,
        pca_components=6,
        lookback_days=40,
        anchor_count=6,
        anchor_policy="even",
        observable_mode=observable_mode,
        trotter_steps_per_anchor=1,
        coupling_scale=0.7,
        transverse_field=0.5,
        evolution_time=0.5,
        ridge_alpha=10.0,
        target_transform="log",
        seed=42,
    )
    print(f"Running observable_mode={observable_mode}")
    result = fit_tfim_qrc_regressor(
        sequence_splits,
        config=config,
        target=target,
        verbose=False,
    )
    probe_rows.append(summarize_qrc_result(result))

observable_probe = pd.DataFrame(probe_rows).sort_values("val_rmse")
observable_probe[[
    "observable_mode", "n_reservoir_features",
    "train_rmse", "val_rmse", "test_rmse",
    "train_qlike", "val_qlike", "test_qlike",
    "train_mz_r2", "val_mz_r2", "test_mz_r2",
]]

## 4. Save results

In [ ]:
out_dir = Path("results/tables")
out_dir.mkdir(parents=True, exist_ok=True)

fallback_summary.to_csv(out_dir / "phase2_tfim_qrc_fallback_summary.csv", index=False)
observable_probe.to_csv(out_dir / "phase2_tfim_qrc_observable_probe.csv", index=False)
pca6.explained_variance.to_csv(out_dir / "phase2_qrc_pca6_explained_variance.csv", index=False)

print("Saved QRC prototype outputs to", out_dir)

## 5. Interpretation template

Use after running the notebook:

```text
The fallback TFIM-QRC prototype runs end-to-end with train-only PCA inputs, compressed 40-day temporal memory, exact observable expectations, and a ridge readout on log realized volatility. This validates the architecture interface: data preprocessing, temporal encoding, quantum reservoir feature extraction, classical readout, and Track A metric evaluation. The result should be compared against persistence, HAR/Ridge/ElasticNet, and the PCA-compressed ESN reservoir baseline. Performance is secondary at this milestone; the main purpose is to establish a reproducible QRC implementation path and identify the most useful next design probe.
```